# Week 3 — V1 Data Pipeline: Lapora (Topic 3 Lab)

**Student:** Akma Izzuddin bin Azman · **Project:** Lapora — AI-assisted MBRS XBRL tagging assistant
**Pathway:** A (PDF / TXT) — three official SSM guidance documents, prepared for later retrieval.
**Scope of this notebook:** ingest → inspect → prepare → validate → store → record lineage → repeatable pipeline.
No embeddings, vector index, model or API key is used (those are later steps).

## Step A1 — From Topic 2 requirements to data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| **FR3**: every suggestion carries a citation to its source document and taxonomy version | Source identity, **page**, and version for each guidance passage | S01 MBRS FAQ (PDF) · S02 mTool User Manual v2.4 (TXT) · S03 SSMxT 2022 Architecture Document (TXT) | Official SSM publications, public by URL. Redistribution terms not verified, so raw files stay local. |
| **FR4**: flag items that SSM guidance says belong under an "Other" element | The FAQ rule text about "Other" elements must be findable later | S01 MBRS FAQ | Authoritative wording; must survive chunking intact. |
| **RAI1 / NFR5**: grounded, traceable output; reload a new document version without code changes | Checksums, declared version, processing record | Same sources, plus the generated manifests | Version labels inside the documents are inconsistent (see inspection), so the declared version is recorded as stated and flagged. |

**Not covered here:** the SSMxT *concept list* itself (needed for FR2) is published as XBRL/XML taxonomy files, which none of the six lab pathways parses. It is the planned next source.

**Before running in Colab:** upload the three source files into `/content/week3_project/data/raw/`
(`FAQ-MBRS.pdf`, `User_manual_mTool_v2_FS2016.pdf.txt`, `SSMxT2022_Architecture_Document.pdf.txt`), then run all cells in order.

## Step A2 — Workspace (Cells 0, 1, 3)

In [1]:
# Cell 0 — PDF library (Colab). Skipped automatically when pypdf is already installed.
try:
    import pypdf
    print("pypdf available:", pypdf.__version__)
except ImportError:
    !pip -q install pypdf

pypdf available: 6.17.0


In [2]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [3]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR
files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)
assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 3
FAQ-MBRS.pdf bytes: 781066
SSMxT2022_Architecture_Document.pdf.txt bytes: 197583
User_manual_mTool_v2_FS2016.pdf.txt bytes: 74226
Upload verified. Select your relevant input(s) for Step A3.


## Common helpers (Cells 4 and 4A, as provided)

In [4]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [5]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


## Lapora source register and Pathway A functions

One configuration per source. **Part B and Part C both read `SOURCES`**, so the same preparation configuration is used in both.

Decisions made after inspecting the real files (evidence is printed in the next cells):
- **Page numbers are kept.** The lab's basic splitter sets `page = None`, but FR3 needs a citable page. PDF pages come from the PDF itself; for the two `.txt` conversions, page numbers are recovered from the repeated page header.
- **Repeated page headers are removed** only after confirming they are boilerplate, and the rule is recorded per source.
- **Table-of-contents pages are flagged (`is_toc`), not deleted**, because their dotted-leader lines would otherwise look like content.

In [6]:
# Cell A0 — Lapora source register + shared Pathway A functions
import re
from collections import Counter
from pypdf import PdfReader

CHUNK_SIZE = 500
METHOD = "paragraph-page"      # selected in Step B3 (compare with "fixed-whole" and "paragraph-whole")

SOURCES = {
    "S01": {
        "slug": "faq",
        "relative_path": "FAQ-MBRS.pdf",
        "title": "MBRS Frequently Asked Questions",
        "owner": "Suruhanjaya Syarikat Malaysia (SSM)",
        "url": "https://www.ssm.com.my/Pages/Register_Business_Company_LLP/Company/document/FAQ-MBRS.pdf",
        "page_source": "pdf-native",
        "page_marker": None,
        "header_regexes": [
            r"^Malaysian Business Reporting System \(MBRS\) FAQs$",
            r"^Version \d+\.\d+ \w+ \d{4} \d+$",
        ],
        "version_regex": r"Version\s+[\d.]+\s+\w+\s+\d{4}",
        "declared_version": "Version 2.0 Jun 2022 (file title: update v2 31052022); 39 of 42 page headers read 'Version 1.0 Jun 2022'",
        "access_rights": "Public SSM document (URL). Redistribution terms not verified: raw file not committed.",
    },
    "S02": {
        "slug": "mtool",
        "relative_path": "User_manual_mTool_v2_FS2016.pdf.txt",
        "title": "mTool User Manual - Financial Statements (text conversion of the PDF)",
        "owner": "Suruhanjaya Syarikat Malaysia (SSM)",
        "url": "https://www.ssm.com.my/Pages/Register_Business_Company_LLP/Company/document/User_manual_mTool_v2_FS2016.pdf",
        "page_source": "txt-page-header",
        "page_marker": r"^User Manual mTool [–-] Financial Statements \(FS\)[ \t]*\nVersion [\d.]+ \w+ \d{4} Page \| (\d+)[ \t]*$",
        "header_regexes": [],
        "version_regex": r"Version\s+[\d.]+\s+\w+\s+\d{4}",
        "declared_version": "Version 2.4 September 2024",
        "access_rights": "Public SSM document (URL). This is a text conversion supplied by the student, not the original PDF. Redistribution terms not verified: raw file not committed.",
    },
    "S03": {
        "slug": "taxonomy_arch",
        "relative_path": "SSMxT2022_Architecture_Document.pdf.txt",
        "title": "SSMxT 2022 Architecture Document (text conversion of the PDF)",
        "owner": "Suruhanjaya Syarikat Malaysia (SSM)",
        "url": "https://www.ssm.com.my/Pages/Register_Business_Company_LLP/Company/document/SSMxT2022_Architecture_Document.pdf",
        "page_source": "txt-page-header",
        "page_marker": r"^SSMxT_2022 Architecture Document[ \t]*\n[ \t]*(\d{1,3})[ \t]*$",
        "header_regexes": [],
        "version_regex": r"SSMxT_2022\s?v\d+\.\d+",
        "declared_version": "Taxonomy SSMxT_2022 v1.0 (as stated in the document preface); document date not stated",
        "access_rights": "Public SSM document (URL). This is a text conversion supplied by the student, not the original PDF. Redistribution terms not verified: raw file not committed.",
    },
}
for _sid, _c in SOURCES.items():
    _c["source_id"] = _sid
    _c["chunk_size"] = CHUNK_SIZE
    _c["method"] = METHOD

def split_txt_pages(text, marker):
    if not marker:
        return [(1, text)]
    ms = list(re.finditer(marker, text, flags=re.M))
    if not ms:
        raise ValueError("Page marker not found - check the text conversion")
    pages = [(1, text[:ms[0].start()])]          # text before the first header = cover page
    for k, m in enumerate(ms):
        end = ms[k + 1].start() if k + 1 < len(ms) else len(text)
        pages.append((int(m.group(1)), text[m.end():end]))
    return pages

def raw_pages(path, cfg):
    path = Path(path)
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
    if path.suffix.lower() == ".txt":
        return split_txt_pages(path.read_text(encoding="utf-8"), cfg.get("page_marker"))
    raise ValueError("Pathway A needs a PDF or TXT source")

def clean_page_text(text, header_regexes=()):
    # Only justified noise is removed: repeated page-header lines (rules are recorded per source),
    # runs of spaces/tabs and surplus blank lines. Content lines are never dropped.
    lines = [re.sub(r"[ \t]+", " ", ln).strip() for ln in text.splitlines()]
    kept = [ln for ln in lines if not any(re.match(rx, ln) for rx in header_regexes)]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(kept)).strip()

def extract_pages(path, cfg):
    return [(n, clean_page_text(t, cfg.get("header_regexes", ()))) for n, t in raw_pages(path, cfg)]

def page_is_toc(text):
    lines = [l for l in text.splitlines() if l.strip()]
    return bool(lines) and sum(bool(re.search(r"\.{5,}", l)) for l in lines) / len(lines) >= 0.4

def fixed_chunks(s, size=500, stats=None):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    if stats is not None:
        for i in range(size, len(s), size):
            if s[i - 1].isalnum() and s[i].isalnum():
                stats["midword"] += 1
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=500, stats=None):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts, current = [], ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            if stats is not None:
                stats["oversize"] += 1
            parts.extend(fixed_chunks(segment, size, stats))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

def chunk_pages(pages, size, method, stats=None):
    out = []
    if method in ("fixed-whole", "paragraph-whole"):
        text = "\n\n".join(t for _, t in pages)
        fn = fixed_chunks if method == "fixed-whole" else paragraph_chunks
        out = [{"page": None, "is_toc": None, "text": c} for c in fn(text, size, stats)]
    elif method == "paragraph-page":
        for n, t in pages:
            toc = page_is_toc(t)
            out += [{"page": n, "is_toc": toc, "text": c} for c in paragraph_chunks(t, size, stats)]
    else:
        raise ValueError("Choose 'fixed-whole', 'paragraph-whole' or 'paragraph-page'")
    return [c for c in out if c["text"].strip()]

def build_records(path, cfg):
    path = Path(path)
    chunks = chunk_pages(extract_pages(path, cfg), cfg["chunk_size"], cfg["method"])
    if not chunks:
        raise ValueError("No text chunks produced")
    sha = source_hash(path)
    return [
        {
            "record_id": f"{cfg['source_id']}-{i:04d}",
            "source_id": cfg["source_id"],
            "source_file": path.name,
            "source_sha256": sha,
            "source_version": cfg["declared_version"],
            "page": c["page"],
            "is_toc": c["is_toc"],
            "processing": f"{cfg['method']}-char-{cfg['chunk_size']}",
            "chars": len(c["text"]),
            "text": c["text"],
        }
        for i, c in enumerate(chunks, 1)
    ]

def write_jsonl(target, records):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open("w", encoding="utf-8") as fh:
        for row in records:
            fh.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Source register ready:", {k: v["relative_path"] for k, v in SOURCES.items()})

Source register ready: {'S01': 'FAQ-MBRS.pdf', 'S02': 'User_manual_mTool_v2_FS2016.pdf.txt', 'S03': 'SSMxT2022_Architecture_Document.pdf.txt'}


## Step A3 — Inspect the selected sources

File existence is not evidence of ingestion. For each source: page count, extracted characters, pages with almost no text, repeated header lines, table-of-contents pages, version strings found, and a short content sample.

In [7]:
# Cell A3 — Inspection evidence for S01, S02, S03
rows = []
for sid, cfg in SOURCES.items():
    f = select_source(relative_path=cfg["relative_path"])
    pages = raw_pages(f, cfg)
    nums = [n for n, _ in pages]
    low = [n for n, t in pages if len(t.strip()) < 100]
    toc = [n for n, t in pages if page_is_toc(t)]
    cleaned = extract_pages(f, cfg)
    removed = sum(len(t) for _, t in pages) - sum(len(t) for _, t in cleaned)
    versions = Counter(re.findall(cfg["version_regex"], "\n".join(t for _, t in pages)))
    first_content = next(t for n, t in cleaned if len(t) >= 500 and not page_is_toc(t))
    assert sum(len(t.strip()) for _, t in pages) > 0, "No usable text: check extraction or OCR needs."
    print(f"--- {sid} {f.name}")
    print("  extraction:", cfg["page_source"], "| pages/segments:", len(pages),
          "| page numbers strictly increasing:", nums == sorted(set(nums)))
    print("  characters:", sum(len(t) for _, t in pages), "| pages with <100 chars:", low)
    print("  table-of-contents pages (dotted leaders):", toc)
    print("  version strings found:", dict(versions))
    print("  characters removed by header rules + whitespace cleanup:", removed)
    print("  content sample:", repr(first_content[:110]))
    rows.append({"source": sid, "file": f.name, "pages": len(pages), "chars": sum(len(t) for _, t in pages),
                 "low_text_pages": low, "toc_pages": toc})

--- S01 FAQ-MBRS.pdf
  extraction: pdf-native | pages/segments: 43 | page numbers strictly increasing: True
  characters: 58285 | pages with <100 chars: [1]
  table-of-contents pages (dotted leaders): [2, 3, 4]
  version strings found: {'Version 2.0 Jun 2022': 3, 'Version 1.0 Jun 2022': 39}
  characters removed by header rules + whitespace cleanup: 6312
  content sample: '1. MBRS filing requirements\n1.1 What is Malaysian Business Reporting System (MBRS)?\n\nThe Malaysian Business Re'
--- S02 User_manual_mTool_v2_FS2016.pdf.txt
  extraction: txt-page-header | pages/segments: 105 | page numbers strictly increasing: True
  characters: 63881 | pages with <100 chars: [24, 25, 26, 46, 98]
  table-of-contents pages (dotted leaders): [2, 3, 4, 5]
  version strings found: {}
  characters removed by header rules + whitespace cleanup: 889
  content sample: '1 Introduction to the MBRS preparation tool (mTool)\nAs part of the SSM XBRL initiative, SSM has made available'
--- S03 SSMxT2022_Architect

## Step B3 — Compare preparation approaches on the same input

Three configurations, all with chunk size 500 characters:
- **fixed-whole** — fixed character slices over the whole document (baseline; page lost)
- **paragraph-whole** — paragraph-aware over the whole document (the lab's basic splitter; page lost)
- **paragraph-page** — paragraph-aware **within each page**, so every chunk keeps its page (chosen if the evidence supports it)

Measures: chunk count, size, chunks starting with a lowercase letter (a sign of starting mid-sentence), chunks ending at punctuation, mid-word cuts, paragraphs longer than the limit (these fall back to fixed slicing), and how many chunks carry a page number.

In [8]:
# Cell 5A1 — Compare fixed-size vs paragraph-aware vs page-aware chunking
import pandas as pd

def chunk_metrics(chunks, stats):
    texts = [c["text"] for c in chunks]
    lens = sorted(len(t) for t in texts)
    n = len(texts)
    return {
        "chunks": n,
        "median_chars": lens[n // 2],
        "max_chars": lens[-1],
        "start_lowercase": sum(1 for t in texts if t.lstrip()[:1].islower()),
        "end_at_punct": sum(1 for t in texts if t.rstrip()[-1:] in ".?!:;)\"”"),
        "midword_cuts": stats["midword"],
        "oversize_paras": stats["oversize"],
        "with_page": sum(1 for c in chunks if c["page"] is not None),
    }

comparison = []
for sid, cfg in SOURCES.items():
    pages = extract_pages(select_source(relative_path=cfg["relative_path"]), cfg)
    for method in ("fixed-whole", "paragraph-whole", "paragraph-page"):
        stats = {"midword": 0, "oversize": 0}
        chunks = chunk_pages(pages, CHUNK_SIZE, method, stats)
        comparison.append({"source": sid, "method": method, **chunk_metrics(chunks, stats)})
cmp_df = pd.DataFrame(comparison)
print(cmp_df.to_string(index=False))

# Accepted limitation of page-aware chunking: a sentence that continues onto the next page is split.
print()
for sid, cfg in SOURCES.items():
    pages = extract_pages(select_source(relative_path=cfg["relative_path"]), cfg)
    cont = [n for n, t in pages if t[:1].islower()]
    print(f"{sid}: pages that begin mid-sentence (continuation from previous page): {len(cont)} of {len(pages)}")

source          method  chunks  median_chars  max_chars  start_lowercase  end_at_punct  midword_cuts  oversize_paras  with_page
   S01     fixed-whole     105           500        500               78            12            66               0          0
   S01 paragraph-whole     129           426        500               22            70            11              11          0
   S01  paragraph-page     135           416        500               21            77            11              11        135
   S02     fixed-whole     127           500        500               93            19            67               0          0
   S02 paragraph-whole     170           460        500               61            77            45              47          0
   S02  paragraph-page     176           427        500               61            83            45              47        176
   S03     fixed-whole     373           500        500              287            21           237    

In [9]:
# Cell 5A1b — Look at actual chunks (short previews) for S01, the FAQ
f = select_source(relative_path=SOURCES["S01"]["relative_path"])
pages = extract_pages(f, SOURCES["S01"])
fixed = chunk_pages(pages, CHUNK_SIZE, "fixed-whole")
paged = chunk_pages(pages, CHUNK_SIZE, "paragraph-page")
k = 40
print("fixed-whole chunk", k, "boundary:")
print("  ends  :", repr(fixed[k]["text"][-45:]))
print("  next  :", repr(fixed[k + 1]["text"][:45]))
sample = next(c for c in paged if c["page"] and c["page"] >= 20 and not c["is_toc"])
print("\nparagraph-page example -> page", sample["page"], "| chars", len(sample["text"]))
print("  starts:", repr(sample["text"][:70]))
print("  ends  :", repr(sample["text"][-50:]))

fixed-whole chunk 40 boundary:
  ends  : 'tudent\n\nOffice 2016 – Home & Student\n\nOffice '
  next  : '365\n\nNote: For instructions on how to check t'

paragraph-page example -> page 20 | chars 162
  starts: '2.2.9 On opening the tool, following error message appears\nError: Cust'
  ends  : 'application domain\ncould not be created\n\nSolution:'


## Step B2/B3 — Export prepared records and validate

Selected configuration: **paragraph-page, 500 characters** (justification in `docs/week3_pipeline_decisions.md`). One prepared JSONL file per source.

In [10]:
# Cell 5A2 — Export traceable text chunks (one file per source)
prepared_paths = {}
for sid, cfg in SOURCES.items():
    f = select_source(relative_path=cfg["relative_path"])
    records = build_records(f, cfg)
    prepared_path = PROCESSED_DIR / f"prepared_text_{sid}_{cfg['slug']}.jsonl"
    write_jsonl(prepared_path, records)

    exported = [json.loads(line) for line in prepared_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(exported) == len(records), "Exported record count does not match"
    assert all(r["text"].strip() for r in exported), "One or more chunks are empty"
    assert all(r["page"] is not None for r in exported), "A chunk lost its page"
    prepared_paths[sid] = prepared_path
    print(f"{sid}: {len(exported)} chunks -> {prepared_path.name} ({prepared_path.stat().st_size} bytes)")
    meta = {k: v for k, v in exported[10].items() if k != "text"}
    print("   metadata of record 11:", meta)
print("PATHWAY A2 completed successfully.")

S01: 135 chunks -> prepared_text_S01_faq.jsonl (105173 bytes)
   metadata of record 11: {'record_id': 'S01-0011', 'source_id': 'S01', 'source_file': 'FAQ-MBRS.pdf', 'source_sha256': 'ef1fb2647ca1becc2491df33193294af7e242615bb5b6ff8f73a785f08d61c99', 'source_version': "Version 2.0 Jun 2022 (file title: update v2 31052022); 39 of 42 page headers read 'Version 1.0 Jun 2022'", 'page': 3, 'is_toc': True, 'processing': 'paragraph-page-char-500', 'chars': 500}
S02: 176 chunks -> prepared_text_S02_mtool.jsonl (122292 bytes)
   metadata of record 11: {'record_id': 'S02-0011', 'source_id': 'S02', 'source_file': 'User_manual_mTool_v2_FS2016.pdf.txt', 'source_sha256': '157e9fabaf783e2f404563ce8036666abb065628fedb46982d2766ab1f0d1db8', 'source_version': 'Version 2.4 September 2024', 'page': 3, 'is_toc': True, 'processing': 'paragraph-page-char-500', 'chars': 142}
S03: 433 chunks -> prepared_text_S03_taxonomy_arch.jsonl (363378 bytes)
   metadata of record 11: {'record_id': 'S03-0011', 'source_id': 

In [11]:
# B3 BASIC OUTPUT CONTRACT — run after the pathway, for each prepared output
for sid, prepared_path in prepared_paths.items():
    assert prepared_path.exists(), 'Prepared output missing.'
    assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
    print(sid, 'Output:', prepared_path.name, 'Size:', prepared_path.stat().st_size)
print('Validation: files exist and are non-empty; content was also inspected above.')

S01 Output: prepared_text_S01_faq.jsonl Size: 105173
S02 Output: prepared_text_S02_mtool.jsonl Size: 122292
S03 Output: prepared_text_S03_taxonomy_arch.jsonl Size: 363378
Validation: files exist and are non-empty; content was also inspected above.


In [12]:
# Extra content check — a non-empty file is not proof of correct content.
# Page numbers must stay inside each source's real range, and the FAQ rule for "Other" elements must be findable.
for sid, prepared_path in prepared_paths.items():
    recs = [json.loads(l) for l in prepared_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    pages = [r["page"] for r in recs]
    print(f"{sid}: chunk pages {min(pages)}..{max(pages)} | table-of-contents chunks: {sum(bool(r['is_toc']) for r in recs)} of {len(recs)}")
faq = [json.loads(l) for l in prepared_paths["S01"].read_text(encoding="utf-8").splitlines() if l.strip()]
hits = [r for r in faq if "“Other”" in r["text"] and not r["is_toc"]]
print("FAQ chunks mentioning the “Other” element rule:", [(r["record_id"], "page", r["page"]) for r in hits])
assert hits, "The FR4 'Other element' guidance was not found in the prepared FAQ chunks"

S01: chunk pages 1..43 | table-of-contents chunks: 21 of 135
S02: chunk pages 1..105 | table-of-contents chunks: 19 of 176
S03: chunk pages 1..111 | table-of-contents chunks: 26 of 433
FAQ chunks mentioning the “Other” element rule: [('S01-0073', 'page', 25)]


## Step B4 — Register lineage, write and verify the manifest (Cells 4.1, 6A, 6B)

In [13]:
# B4.1 — Register each selected source -> prepared output
for sid, cfg in SOURCES.items():
    register_lineage(
        source=RAW_DIR / cfg["relative_path"],
        output=prepared_paths[sid],
        processing=(f"{cfg['method']} text chunks of {cfg['chunk_size']} characters; "
                    f"page recovery={cfg['page_source']}; header rules={len(cfg['header_regexes'])}; "
                    "table-of-contents pages flagged, not deleted")
    )

Registered: FAQ-MBRS.pdf -> prepared_text_S01_faq.jsonl
Registered: User_manual_mTool_v2_FS2016.pdf.txt -> prepared_text_S02_mtool.jsonl
Registered: SSMxT2022_Architecture_Document.pdf.txt -> prepared_text_S03_taxonomy_arch.jsonl


In [14]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# Adapted: source_version / access_rights come from the SOURCES register instead of "Not verified".
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)
by_file = {c["relative_path"]: c for c in SOURCES.values()}
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]
    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    meta = by_file[str(source.relative_to(RAW_DIR.resolve()))]
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "source_url": meta["url"],
        "source_owner": meta["owner"],
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": meta["declared_version"] + " [as declared in the document; not independently verified with SSM]",
        "access_rights": meta["access_rights"],
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 3
Saved to: /content/week3_project/data/week3_manifest.json


In [15]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 3
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Executable V1 data pipeline

Cell 7 (common utilities) is as provided. Cell 8A is the Pathway A pipeline: it **re-uses the Part B functions and the same `SOURCES` configuration**, and runs once per source so each source gets its own folder and `pipeline_manifest.json`.

In [16]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )
    return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )
        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified — check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)
    return record

# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )
    first = runner()
    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [17]:
# Cell 8A — Pathway A (PDF/TXT) pipeline, one run per selected source
# Uses the Part B functions and the same SOURCES configuration.
def inspect_documents(inputs, cfg):
    src = inputs[0]
    pages = raw_pages(src, cfg)
    total = sum(len(t.strip()) for _, t in pages)
    if total == 0:
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    nums = [n for n, _ in pages]
    if nums != sorted(set(nums)):
        raise ValueError("Page numbers are not strictly increasing - page recovery failed")
    return {
        "source": src.name,
        "pages": len(pages),
        "characters": total,
        "low_text_pages": [n for n, t in pages if len(t.strip()) < 100],
        "toc_pages": [n for n, t in pages if page_is_toc(t)],
    }

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    records = build_records(src, cfg)
    target = out / "prepared_text.jsonl"
    write_jsonl(target, records)
    mapping = [{"source": str(src), "output": str(target)}]
    method = (f"{cfg['method']} text chunks of {cfg['chunk_size']} characters; "
              f"page recovery={cfg['page_source']}; header rules={len(cfg['header_regexes'])}; "
              "table-of-contents pages flagged, not deleted")
    return [target], mapping, method

runs_A = {}
for sid, cfg in SOURCES.items():
    runs_A[sid] = register_and_run(
        f"documents_{cfg['slug']}",
        resolve_source(cfg["relative_path"]),
        inspect_documents,
        prepare_documents,
        cfg,
    )
print("Cell 8A completed successfully.")

SUCCESS: documents_faq | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_faq/pipeline_manifest.json
SUCCESS: documents_mtool | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mtool/pipeline_manifest.json
SUCCESS: documents_taxonomy_arch | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_taxonomy_arch/pipeline_manifest.json
Cell 8A completed successfully.


In [18]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A-8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} — source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents_faq | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_faq/pipeline_manifest.json
PASS: documents_faq — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_mtool | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_mtool/pipeline_manifest.json
PASS: documents_mtool — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_taxonomy_arch | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_taxonomy_arch/pipeline_manifest.json
PASS: documents_taxonomy_arch — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


## Extra checks for the stage contracts (Step C3)

1. **Ingest** stops when the chosen input is missing.
2. **Inspect** stops when a file has no extractable text (instead of passing an empty result downstream).
3. **Consistency with Part B:** the Part C output is byte-identical to the Part B output for the same source.

In [19]:
# Cell 9B — Failure-path and consistency checks
import tempfile

# 1) Ingest contract: a missing input must stop the pipeline
try:
    resolve_source("does_not_exist.pdf")
except FileNotFoundError as e:
    print("PASS - ingest stops on missing input:", e)

# 2) Inspect contract: a file with no extractable text must stop, not be prepared
with tempfile.TemporaryDirectory() as tmp:
    empty = Path(tmp) / "empty.txt"
    empty.write_text("   \n", encoding="utf-8")
    try:
        inspect_documents([empty], {"page_marker": None})
    except ValueError as e:
        print("PASS - inspect stops on empty text:", e)

# 3) Part B output == Part C output for each source
for sid, cfg in SOURCES.items():
    b_out = prepared_paths[sid]
    c_out = PIPE_DIR / f"documents_{cfg['slug']}" / "prepared_text.jsonl"
    same = sha256_file(b_out) == sha256_file(c_out)
    print(f"{sid}: Part B output == Part C output:", same)
    assert same, "Part B and Part C outputs differ - investigate"

PASS - ingest stops on missing input: Source file not found: /content/week3_project/data/raw/does_not_exist.pdf
PASS - inspect stops on empty text: No extractable text found. Scanned PDFs may require OCR.
S01: Part B output == Part C output: True
S02: Part B output == Part C output: True
S03: Part B output == Part C output: True


In [20]:
# Final inventory of generated artefacts
print("Part B prepared outputs:")
for p in sorted(PROCESSED_DIR.glob("*")):
    print("  ", p.relative_to(BASE_DIR), p.stat().st_size, "bytes")
print("Part B manifest:", (BASE_DIR / "data" / "week3_manifest.json").relative_to(BASE_DIR))
print("Part C outputs:")
for p in sorted(PIPE_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(BASE_DIR), p.stat().st_size, "bytes")

Part B prepared outputs:
   data/processed/prepared_text_S01_faq.jsonl 105173 bytes
   data/processed/prepared_text_S02_mtool.jsonl 122292 bytes
   data/processed/prepared_text_S03_taxonomy_arch.jsonl 363378 bytes
Part B manifest: data/week3_manifest.json
Part C outputs:
   data/pipeline_v1/documents_faq/pipeline_manifest.json 2100 bytes
   data/pipeline_v1/documents_faq/prepared_text.jsonl 105173 bytes
   data/pipeline_v1/documents_mtool/pipeline_manifest.json 2312 bytes
   data/pipeline_v1/documents_mtool/prepared_text.jsonl 122292 bytes
   data/pipeline_v1/documents_taxonomy_arch/pipeline_manifest.json 2318 bytes
   data/pipeline_v1/documents_taxonomy_arch/prepared_text.jsonl 363378 bytes
